<div align="center">
  <img src="https://datasupplychain.waverlyrose.io/healthstack-logo-dark.png" alt="HealthStack logo" width="320" />
</div>

<p align="center"><strong>AFROTECH 2026</strong></p>
<p align="center"><font size="6"><strong>Building Fairness Into the Data Supply Chain</strong></font></p>
<p align="center"><font size="5"><strong>Instructor: Waverly Rose Brim</strong></font></p>
<p align="center"><font size="4"><strong>The Hidden Harm Challenge</strong></font></p>

<div align="center">
  <img src="https://datasupplychain.waverlyrose.io/hidden-harm.gif" alt="Two silhouettes stand in green spotlights on a dark stage. Each figure is filled with columns of numbers. In the animated version the spotlights move across the figures." width="640" />
  <p><em>You choose where to look.</em></p>
</div>

Work down the page: **predict → edit → run → explain**.
Later challenges are one edit and one run. Do not use **Run All**.

### **Colab Setup · install**

This is the same package setup as Act II. Nothing to change here: just run the cell.

In [ ]:
import sys, subprocess

if "google.colab" in sys.modules:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade", "--force-reinstall", "--no-deps", "--no-cache-dir",
        "data-supply-chain-workshop==0.1.2",
    ])
else:
    import data_supply_chain

### **Setup · select the runtime**

In Colab, choose **Runtime → Change runtime type → T4 GPU**. If Colab cannot give you a GPU,
keep going: the notebook switches to CPU on its own. CPU runs the same search with the same
results; each correction step just takes a few seconds longer.

In [ ]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    try:
        device = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
        has_gpu = device.returncode == 0
    except FileNotFoundError:
        has_gpu = False
    if not has_gpu:
        print("No GPU attached. Continuing on CPU: same search, same results, a little slower.")
        print("(Optional: Runtime → Change runtime type → T4 GPU, then run setup again.)")

### **Setup · connect the accelerator**

Run this once. The printed backend tells you whether the accelerator is available.

In [ ]:
from data_supply_chain.search import gpu_backend_available

CUPY_FOR_CUDA = {12: "cupy-cuda12x==13.6.0", 13: "cupy-cuda13x==14.0.1"}
SMOKE = "import cupy as cp; cp.asnumpy(cp.arange(4)[cp.asarray([1, 2])] * 2)"

if IN_COLAB and has_gpu:
    import re
    _m = re.search(r"CUDA Version: (\d+)", device.stdout)
    _wheel = CUPY_FOR_CUDA.get(int(_m.group(1))) if _m else None
    if _wheel and subprocess.run([sys.executable, "-c", SMOKE], capture_output=True).returncode != 0:
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                        "cupy-cuda12x", "cupy-cuda13x"])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _wheel])
    try:
        import cupy as _cp
        if not hasattr(_cp, "put_along_axis"):
            _cp.put_along_axis = lambda arr, indices, values, axis=None: arr.__setitem__(
                (_cp.arange(arr.shape[0])[:, None], indices), values
            )
    except ImportError:
        pass
USE_GPU = gpu_backend_available()
if USE_GPU:
    try:
        import cupy as _cp
        _cp.asnumpy(_cp.arange(4)[_cp.asarray([1, 2])] * 2)
    except Exception:
        USE_GPU = False
if IN_COLAB and has_gpu and not USE_GPU:
    print("CuPy cannot use the GPU, so this run uses CPU instead. Results are the same.")
print("GPU ready" if USE_GPU else "CPU path: same search, same results")

### **Cold start**

Same cohort. Same split. Same mortality model as Act II. Earlier notebooks do not need to be open.
If the kernel dies later, run the setup cells and this cell again, then re-run the section you were in.

In [ ]:
from data_supply_chain.bootstrap import cold_start, reset
from data_supply_chain.models import fit_audit_model
from data_supply_chain.preprocess import distribution_variant

S = fit_audit_model(cold_start())
print(f"{len(S.df):,} adults · {len(S.test_idx):,} held-out rows · AUROC {S.auroc:.4f}")
print(f"source={S.source} · real={S.is_real} · distribution={distribution_variant()}")
if not S.is_real:
    print("SYNTHETIC DATA — practice only; do not compare with the site's findings.")

### **Escape hatch · broke something? Run this cell.**

Skip this on your first pass. If a later cell errors or `S` looks wrong, scroll back here and run it.
It rebuilds the same cohort, split, and model, so no number moves. Then re-run the section you were in.
If it says `reset` is not defined, the runtime restarted: run setup and the Cold start cell first.

In [ ]:
S = reset()

### **Load the search tools**

These are the workshop’s existing search functions. We will call them in small steps.
The charts only draw the tables the search returns; they never run another search.

In [ ]:
import pandas as pd
from IPython.display import display
from data_supply_chain.search import last_backend, slice_lattice, permutation_null
from data_supply_chain.audit import EmptyAuditError
from data_supply_chain.hidden_harm_visuals import (
    plot_audit_counts, plot_group_intervals, plot_peer_difference,
    plot_corrected_pvalues, plot_pvalue_change, search_details,
)

### **1.) Make a prediction**

Until now, someone chose the subgroups for you. Which group do you expect to perform worse?

**Feel free to use this section to capture your thoughts** *(double-click this cell to type)*

My prediction:

Why I expect this:

### **EDIT HERE · Choose where to look**

These four settings are the only lines you change. Everything else stays as written.

| Setting | What it controls | What you can set it to |
|---|---|---|
| `AXES` | Which characteristics define the subgroups | A list using any of `"race"`, `"insurance"`, `"diabetic"`, `"female"`, each in quotes and listed once |
| `MAX_DEPTH` | How many characteristics can combine into one group. `1` = each characteristic on its own; `2` adds pairs such as Black & Uninsured | A whole number from `1` up to how many `AXES` you listed |
| `MIN_N` | The smallest group the audit will test. Smaller groups are suppressed, not passed | A whole number above `0` (for example `30`) |
| `ALPHA` | The corrected significance threshold a group must beat to survive | A number between `0` and `1` (for example `0.05` or `0.01`) |

**Run the defaults first.** Challenges 2 and 3 compare their searches with this one. If you type a value that is not allowed, the next cell stops and tells you which setting to fix.

In [ ]:
AXES = ["race", "insurance"]  # Any of: "race", "insurance", "diabetic", "female"
MAX_DEPTH = 2                 # Whole number, 1 up to len(AXES). 1 = no intersections
MIN_N = 30                    # Whole number above 0. Smaller groups are suppressed
ALPHA = 0.05                  # Between 0 and 1, e.g. 0.05 or 0.01

### **Run this search**

Run the next three cells in order. Start by checking the controls and saving this search’s settings.

In [ ]:
baseline_lat = baseline_corrected = None
assert isinstance(AXES, list) and AXES and all(isinstance(a, str) for a in AXES), "Use a list of axis names."
assert len(set(AXES)) == len(AXES) and set(AXES) <= {"race", "insurance", "diabetic", "female"}, "Check axis names."
assert type(MAX_DEPTH) is int and 1 <= MAX_DEPTH <= len(AXES), "Check MAX_DEPTH."
assert type(MIN_N) is int and MIN_N > 0, "MIN_N must be positive."
assert type(ALPHA) in (int, float) and 0 < ALPHA < 1, "ALPHA must be between 0 and 1."
baseline_settings = dict(axes=[a for a in ["race", "insurance", "diabetic", "female"] if a in AXES],
                         max_depth=MAX_DEPTH, min_n=MIN_N, use_gpu=USE_GPU)
baseline_alpha = ALPHA
print("Baseline:", baseline_settings, "alpha=", baseline_alpha)

### **Build the subgroup table**

The model stays fixed. These settings determine which groups we examine.

In [ ]:
baseline_lat = baseline_corrected = None
baseline_lat = slice_lattice(
    S.audit_sub, y=S.y_test, p=S.predictions, **baseline_settings,
)
print(f"{len(baseline_lat)} subgroups considered · backend={last_backend()}")

### **Account for the whole search**

Now compare each subgroup’s raw AUROC with the worst AUROC in each permuted search. If every group is suppressed, there is no family to test.

In [ ]:
baseline_corrected = None
try:
    baseline_corrected = permutation_null(
        S.audit_sub, y=S.y_test, p=S.predictions,
        alpha=baseline_alpha, **baseline_settings,
    )
    print(f"Corrected {len(baseline_corrected)} groups · backend={last_backend()}")
except EmptyAuditError as exc:
    print(exc)

### **Count what made it through**

Considered, auditable, survived. Read these counts before choosing a winner. Suppression can mean too few rows or only one outcome class.

In [ ]:
auditable = int(baseline_lat.suppressed.isna().sum())
print("Considered:", len(baseline_lat))
print("Auditable:", auditable)
print("Suppressed:", len(baseline_lat) - auditable)
print("Survived:", "not evaluated" if baseline_corrected is None else int(baseline_corrected.survives.sum()))

**Read the stages.** Each bar is a stage of the same search, so do not add them. A surviving signal is a flagged result, not a group declared safe.

In [ ]:
plot_audit_counts([("Baseline", baseline_lat, baseline_corrected)])

### **Look inside the groups**

`n_pos` counts deaths. A small group or few outcomes can make AUROC unstable. Inspect the interval and suppression reason alongside the point estimate.

In [ ]:
display(baseline_lat[["cell", "n", "n_pos", "auroc", "lo", "hi", "suppressed"]].round(4))

**Read the dot and its line together.** The line is the group's 95% bootstrap interval; each label carries its size and deaths. The dashed overall AUROC is for orientation only. A small group is not measured on the same scale as the whole test set, which is why the next comparison exists.

In [ ]:
plot_group_intervals(baseline_lat, S.auroc)

### **Compare with matched peers**

`excess` is subgroup AUROC minus the mean from reference groups matched on size and outcome counts. Negative means below that reference. It is descriptive, not proof of a cause.

In [ ]:
display(baseline_lat[["cell", "auroc", "ref_auroc", "excess"]].round(4))

**Read each bar against its band.** The grey band is where 95% of the matched reference groups land. A bar inside it is a gap that groups of the same shape show anyway; a bar far outside it is worth a closer look. Still descriptive: the next section is the corrected test.

In [ ]:
plot_peer_difference(baseline_lat)

### **See what survives**

`p_fwer` tests **raw AUROC**, not `excess`. Survival means `p_fwer < ALPHA` for this search. No survivors does not establish safety.

In [ ]:
if baseline_corrected is None:
    print("No auditable family — insufficient evidence.")
else:
    display(baseline_corrected[["cell", "n", "auroc", "p_fwer", "survives"]])

**Read the rule.** The dashed line is the alpha this search was corrected at, read from the result itself, so it cannot drift if you edit `ALPHA` later. "Does not survive" does not mean safe.

In [ ]:
plot_corrected_pvalues(baseline_corrected)

### **What was the cost of looking?**

The null tracks the worst AUROC in each permuted search. `null_min` is the lowest simulated minimum, not an alpha cutoff. Interpretation depends on joint subgroup-label exchangeability; this is an exploratory cohort audit.

In [ ]:
search_details(baseline_corrected, baseline_settings)

### **Your first interpretation**

Find the insurance-related rows and then the race-only rows. Does a quiet AUROC result erase the threshold consequence from Inherited Decisions?

**Feel free to use this section to capture your thoughts** *(double-click this cell to type)*

The group I would investigate next:

The evidence I would cite:

The kind of harm this AUROC search cannot detect:

### **2.) Widen the search · predict**

Add `"diabetic"` while keeping the other defaults. Predict what happens to the number of groups and the corrected p-values before changing the cell below.

**Feel free to use this section to capture your thoughts** *(double-click this cell to type)*

I expect the number of groups to:

I expect the correction to:

### **Your turn · add one axis**

Add `"diabetic"` to `AXES` here, then run the next cell.

In [ ]:
AXES = ["race", "insurance"]  # EDIT: add "diabetic" → ["race", "insurance", "diabetic"]
MAX_DEPTH = 2                 # Keep 2 (pairs only)
MIN_N = 30                    # Keep 30
ALPHA = 0.05                  # Keep 0.05

### **Run and compare**

One cell: build the table, pay for the search, then compare with the default.

In [ ]:
wider_lat = wider_corrected = None
assert isinstance(AXES, list) and AXES and all(isinstance(a, str) for a in AXES), "Use a list of axis names."
assert len(set(AXES)) == len(AXES) and set(AXES) <= {"race", "insurance", "diabetic", "female"}, "Check axis names."
assert type(MAX_DEPTH) is int and 1 <= MAX_DEPTH <= len(AXES), "Check MAX_DEPTH."
assert type(MIN_N) is int and MIN_N > 0, "MIN_N must be positive."
assert type(ALPHA) in (int, float) and 0 < ALPHA < 1, "ALPHA must be between 0 and 1."
wider_settings = dict(axes=[a for a in ["race", "insurance", "diabetic", "female"] if a in AXES],
                      max_depth=MAX_DEPTH, min_n=MIN_N, use_gpu=USE_GPU)
wider_alpha = ALPHA
print("Wider search:", wider_settings, "alpha=", wider_alpha)
if wider_settings["axes"] == baseline_settings["axes"]:
    print("This search still matches the default. Add \"diabetic\" to AXES above, then re-run this cell.")
else:
    wider_lat = slice_lattice(
        S.audit_sub, y=S.y_test, p=S.predictions, **wider_settings,
    )
    print(f"{len(wider_lat)} subgroups considered · backend={last_backend()}")
    try:
        wider_corrected = permutation_null(
            S.audit_sub, y=S.y_test, p=S.predictions,
            alpha=wider_alpha, **wider_settings,
        )
        print(f"Corrected {len(wider_corrected)} groups · backend={last_backend()}")
    except EmptyAuditError as exc:
        print(exc)
    display(pd.DataFrame({
        "baseline": [len(baseline_lat), int(baseline_lat.suppressed.isna().sum())],
        "wider": [len(wider_lat), int(wider_lat.suppressed.isna().sum())],
    }, index=["considered", "auditable"]))
    if baseline_corrected is None or wider_corrected is None:
        print("Comparison unavailable: at least one search had no auditable family.")
    else:
        cols = ["cell", "auroc", "p_fwer", "survives"]
        display(baseline_corrected[cols].merge(
            wider_corrected[cols], on="cell", suffixes=("_baseline", "_wider"),
        ))

**Compare one row at a time.** Circle = baseline, square = wider search, for the same group. Its AUROC in both searches is printed under its name; its p-values and decisions are on the right. Only groups both searches tested appear.

In [ ]:
plot_pvalue_change(baseline_corrected, wider_corrected, "Baseline", "Wider search",
                   title="Widen the search: did the AUROC move, or the correction?")

### **Explain the change**

The patients and predictions stayed fixed. What changed when you gave yourself more places to look? Report what you observed, even if no survival decision changed.

**Feel free to use this section to capture your thoughts** *(double-click this cell to type)*

The result that changed:

The result that stayed the same:

My explanation:

### **3.) Remove intersections · predict**

Return to race and insurance. What do you expect when the search tests each axis separately, without their intersections?

**Feel free to use this section to capture your thoughts** *(double-click this cell to type)*

My prediction for the family size:

My prediction for a shared group's AUROC and p_fwer:

### **Your turn · change the depth**

Change `MAX_DEPTH` to `1`, then run the next cell.

In [ ]:
AXES = ["race", "insurance"]  # Keep: back to race and insurance only
MAX_DEPTH = 2                 # EDIT: change to 1 (each characteristic on its own)
MIN_N = 30                    # Keep 30
ALPHA = 0.05                  # Keep 0.05

### **Run and compare**

The patients are identical. The predictions are identical. Only the size of the search family should change.

In [ ]:
shallow_lat = shallow_corrected = None
assert isinstance(AXES, list) and AXES and all(isinstance(a, str) for a in AXES), "Use a list of axis names."
assert len(set(AXES)) == len(AXES) and set(AXES) <= {"race", "insurance", "diabetic", "female"}, "Check axis names."
assert type(MAX_DEPTH) is int and 1 <= MAX_DEPTH <= len(AXES), "Check MAX_DEPTH."
assert type(MIN_N) is int and MIN_N > 0, "MIN_N must be positive."
assert type(ALPHA) in (int, float) and 0 < ALPHA < 1, "ALPHA must be between 0 and 1."
shallow_settings = dict(axes=[a for a in ["race", "insurance", "diabetic", "female"] if a in AXES],
                        max_depth=MAX_DEPTH, min_n=MIN_N, use_gpu=USE_GPU)
shallow_alpha = ALPHA
print("Single-axis search:", shallow_settings, "alpha=", shallow_alpha)
if shallow_settings["max_depth"] != 1:
    print("MAX_DEPTH is still 2. Change it to 1 above, then re-run this cell.")
else:
    shallow_lat = slice_lattice(
        S.audit_sub, y=S.y_test, p=S.predictions, **shallow_settings,
    )
    print(f"{len(shallow_lat)} subgroups considered · backend={last_backend()}")
    try:
        shallow_corrected = permutation_null(
            S.audit_sub, y=S.y_test, p=S.predictions,
            alpha=shallow_alpha, **shallow_settings,
        )
        print(f"Corrected {len(shallow_corrected)} groups · backend={last_backend()}")
    except EmptyAuditError as exc:
        print(exc)
    if baseline_corrected is None or shallow_corrected is None:
        print("Comparison unavailable: at least one search had no auditable family.")
    else:
        cols = ["cell", "auroc", "p_fwer", "survives"]
        display(baseline_corrected[cols].merge(
            shallow_corrected[cols], on="cell", suffixes=("_baseline", "_shallow"),
        ))

**Same chart, other direction.** The intersections are gone from the single-axis search. They were not tested, which is neither a pass nor a fail, so they are left off.

In [ ]:
plot_pvalue_change(baseline_corrected, shallow_corrected, "Baseline", "Single-axis search",
                   title="Remove intersections: compare the shared groups")

### **Did the evidence move?**

For a group present in both searches, did the underlying AUROC change, or did the number of questions change? A p-value need not move for every group.

**Feel free to use this section to capture your thoughts** *(double-click this cell to type)*

A shared subgroup:

What changed:

What this tells me about reporting only a winner:

### **Optional · make the lattice too thin**

Four axes, three-way intersections, `MIN_N = 30`. Run the next cell as-is. At this threshold only a few groups fail the size rule — the quieter failure is a wide interval on a small auditable group.

**Feel free to use this section to capture your thoughts** *(double-click this cell to type)*

My prediction for how many groups will be suppressed:

In [ ]:
# Already set. Run as-is, no edits needed.
AXES = ["race", "insurance", "diabetic", "female"]  # All four characteristics
MAX_DEPTH = 3                                       # Up to three-way intersections
MIN_N = 30                                          # Same size rule as before
ALPHA = 0.05

### **Where does the evidence run out?**

An untestable group is not a group that passed. Read the suppressed rows, the smallest auditable intervals, then what survives correction.

In [ ]:
thin_lat = thin_corrected = None
assert isinstance(AXES, list) and AXES and all(isinstance(a, str) for a in AXES), "Use a list of axis names."
assert len(set(AXES)) == len(AXES) and set(AXES) <= {"race", "insurance", "diabetic", "female"}, "Check axis names."
assert type(MAX_DEPTH) is int and 1 <= MAX_DEPTH <= len(AXES), "Check MAX_DEPTH."
assert type(MIN_N) is int and MIN_N > 0, "MIN_N must be positive."
assert type(ALPHA) in (int, float) and 0 < ALPHA < 1, "ALPHA must be between 0 and 1."
thin_settings = dict(axes=[a for a in ["race", "insurance", "diabetic", "female"] if a in AXES],
                     max_depth=MAX_DEPTH, min_n=MIN_N, use_gpu=USE_GPU)
thin_alpha = ALPHA
print("Deeper intersections:", thin_settings, "alpha=", thin_alpha)
thin_lat = slice_lattice(
    S.audit_sub, y=S.y_test, p=S.predictions, **thin_settings,
)
print(f"{len(thin_lat)} subgroups considered · backend={last_backend()}")
n_suppressed = int(thin_lat.suppressed.notna().sum())
print("Auditable:", len(thin_lat) - n_suppressed)
print("Suppressed:", n_suppressed, "of", len(thin_lat))
print("Suppressed rows:")
display(thin_lat.loc[thin_lat.suppressed.notna(), ["cell", "n", "n_pos", "suppressed"]])
small = thin_lat.loc[thin_lat.suppressed.isna()].nsmallest(8, "n")
print("Smallest auditable groups — watch interval width:")
display(small[["cell", "n", "n_pos", "auroc", "lo", "hi"]].round(4))
if not IN_COLAB:
    print("Correction skipped on local/CI CPU. On a Colab T4 this cell also pays for the search.")
else:
    try:
        thin_corrected = permutation_null(
            S.audit_sub, y=S.y_test, p=S.predictions,
            alpha=thin_alpha, **thin_settings,
        )
        details = thin_corrected.attrs
        print(f"Corrected {len(thin_corrected)} groups · backend={last_backend()}")
        print("Family size:", details["n_family"])
        print("Lowest simulated minimum:", round(details["null_min"], 4))
        print("Survived:", int(thin_corrected.survives.sum()))
        display(thin_corrected.loc[thin_corrected.survives, ["cell", "n", "auroc", "p_fwer", "survives"]])
    except EmptyAuditError as exc:
        print(exc)

**Read the evidence limit.** These are the eight smallest auditable groups from the table above, smallest first. Each passed the size rule; look at how wide its interval still is.

In [ ]:
plot_group_intervals(small, S.auroc, sort_by=None,
                     title="Smallest auditable groups: a number is not precise evidence")

### **Optional · lower the minimum**

The tempting fix is to lower `MIN_N` until more groups appear. That does not create evidence; it removes the check that was telling you there was none. This cell is already set to `5`. Predict, then run.

**Feel free to use this section to capture your thoughts** *(double-click this cell to type)*

I expect auditable groups, the null minimum, and survivors to:

In [ ]:
# Already set. Run as-is, no edits needed.
AXES = ["race", "insurance", "diabetic", "female"]  # Same as the cell above
MAX_DEPTH = 3                                       # Same as the cell above
MIN_N = 5                                           # The only change: lowered from 30
ALPHA = 0.05

### **Pay for the smaller groups**

Compare with the `MIN_N = 30` search above. You bought more opportunities to find something extreme. The search charges you for them.

In [ ]:
thin5_lat = thin5_corrected = None
thin5_settings = dict(axes=[a for a in ["race", "insurance", "diabetic", "female"] if a in AXES],
                      max_depth=MAX_DEPTH, min_n=MIN_N, use_gpu=USE_GPU)
print("Lowered minimum:", thin5_settings, "alpha=", ALPHA)
thin5_lat = slice_lattice(
    S.audit_sub, y=S.y_test, p=S.predictions, **thin5_settings,
)
print(f"{len(thin5_lat)} subgroups considered · backend={last_backend()}")
if "thin_lat" not in dir() or thin_lat is None:
    print("Run the MIN_N = 30 cell first so there is something to compare.")
else:
    display(pd.DataFrame({
        "min_n=30": [
            len(thin_lat),
            int(thin_lat.suppressed.isna().sum()),
            int(thin_lat.suppressed.notna().sum()),
        ],
        "min_n=5": [
            len(thin5_lat),
            int(thin5_lat.suppressed.isna().sum()),
            int(thin5_lat.suppressed.notna().sum()),
        ],
    }, index=["considered", "auditable", "suppressed"]))
if not IN_COLAB:
    print("Correction skipped on local/CI CPU. On a Colab T4 this cell also pays for the smaller groups.")
else:
    try:
        thin5_corrected = permutation_null(
            S.audit_sub, y=S.y_test, p=S.predictions,
            alpha=ALPHA, **thin5_settings,
        )
        print(f"Corrected {len(thin5_corrected)} groups · backend={last_backend()}")
        print("Family size:", thin5_corrected.attrs["n_family"])
        print("Lowest simulated minimum:", round(thin5_corrected.attrs["null_min"], 4))
        print("Survived:", int(thin5_corrected.survives.sum()))
        if thin_corrected is not None:
            print(
                "At MIN_N = 30 → family", thin_corrected.attrs["n_family"],
                "· null_min", round(thin_corrected.attrs["null_min"], 4),
                "· survived", int(thin_corrected.survives.sum()),
            )
    except EmptyAuditError as exc:
        print(exc)

**Compare the two searches stage by stage.** Lowering the size rule makes more groups auditable; it does not make them better measured. A correction that did not run shows as "Not evaluated", not zero.

In [ ]:
plot_audit_counts([("MIN_N = 30", globals().get("thin_lat"), globals().get("thin_corrected")),
                   ("MIN_N = 5", thin5_lat, thin5_corrected)],
                  title="Lower the minimum: what changed?")

**Feel free to use this section to capture your thoughts** *(double-click this cell to type)*

My interpretation:

### **What will you ask of your own audit?**

How many groups were searched before the reported one was chosen? Could the metric detect the harm you care about? Repeatedly searching and choosing a winner is not covered by one run’s correction.
If you have time, rerun the default search with `ALPHA = 0.01` and ask which moved: the AUROCs, the p-values, or the survival rule.

**Feel free to use this section to capture your thoughts** *(double-click this cell to type)*

One question I will ask about the search:

One question I will ask about the metric:

**Completed the challenge?**

[Continue to the go/no-go checklist](https://datasupplychain.waverlyrose.io/11-checklist/)

### **Record**

Skip this. It always records the pinned default search on CPU, not the cells you edited. In Colab it does not write a file.

In [ ]:
from data_supply_chain.findings import Findings
from data_supply_chain.paths import findings_path

_default = slice_lattice(S.audit_sub, ["race", "insurance"], S.y_test, S.predictions,
                         min_n=30, max_depth=2, use_gpu=False)
_default_corrected = permutation_null(S.audit_sub, ["race", "insurance"], S.y_test,
                                      S.predictions, min_n=30, max_depth=2,
                                      use_gpu=False)

f = Findings.open()
f.record("search.default.axes", "race, insurance")
f.record("search.default.enumerated", int(len(_default)))
f.record("search.default.auditable", int(_default.suppressed.isna().sum()))
f.record("search.default.survived", int(_default_corrected.survives.sum()))
f.record("search.default.died_on_size",
         int(len(_default) - _default.suppressed.isna().sum()))
f.record("search.default.died_on_multiplicity",
         int(_default.suppressed.isna().sum() - _default_corrected.survives.sum()))
f.record("search.default.family_n", int(_default_corrected.attrs["n_family"]))
f.record("search.default.null_min", round(float(_default_corrected.attrs["null_min"]), 4))

if findings_path() is None:
    print("Outside a repo checkout — findings.json not written (expected in Colab).")
else:
    f.write(state=S)
    print("wrote findings.json")
print("findings.search came from the DEFAULT search above, never from your edits")